# 03 Flood Susceptibility Model

This notebook combines terrain, hydrology, rainfall, and land-cover factors into a flood susceptibility score and classified risk map.

## What you will do

1. Review the flood-conditioning factors.
2. Check which factor rasters are available.
3. Review model weights.
4. Run the susceptibility workflow.
5. Inspect score, class, and class-area outputs.

## Step 1: Load configuration and factor weights

In [ ]:
from pathlib import Path
import pandas as pd
from floodsense.config import load_config

config = load_config()
paths = config['paths']
weights = config['weights']
pd.DataFrame([{'factor': k, 'weight': v} for k, v in weights.items()])

## Step 2: Understand the model logic

The model normalizes each factor to a 0-1 scale where higher values mean higher flood risk.

Risk direction:

- Lower elevation = higher risk.
- Lower slope = higher risk.
- Higher flow accumulation = higher risk.
- Shorter distance to stream/river = higher risk.
- Higher rainfall = higher risk.
- Built-up, wetland, and water-related land-cover classes can increase risk when recognized.

## Step 3: Check required and optional factor rasters

The workflow uses whatever aligned factor rasters are available. DEM and hydrology outputs are the most important starting point.

In [ ]:
processed = Path(paths['processed_rasters'])
interim = Path(paths['interim_reprojected'])
factor_candidates = {
    'elevation': [processed / 'dem_filled.tif', interim / 'dem.tif'],
    'slope': [processed / 'slope.tif'],
    'flow_accumulation': [processed / 'flow_accumulation.tif'],
    'distance_to_river_or_stream': [processed / 'distance_to_stream.tif', processed / 'distance_to_river.tif'],
    'rainfall': [interim / 'rainfall.tif'],
    'landcover': [interim / 'landcover.tif'],
}

for factor, candidates in factor_candidates.items():
    found = [p for p in candidates if p.exists()]
    print(f'{factor:28s}:', found[0] if found else 'missing')

## Step 4: Run susceptibility workflow

Outputs:

- `data/processed/rasters/susceptibility_score.tif`
- `data/processed/rasters/susceptibility_class.tif`
- `data/processed/tables/susceptibility_class_area.csv`
- `outputs/maps/susceptibility_preview.png`

In [ ]:
from floodsense.susceptibility import run_susceptibility_workflow

run_susceptibility_workflow(config)

## Step 5: Inspect class-area table

In [ ]:
area_table = Path(paths['processed_tables']) / 'susceptibility_class_area.csv'
if area_table.exists():
    display(pd.read_csv(area_table))
else:
    print('Class area table not available yet.')

## Step 6: Generate/refresh preview map

In [ ]:
from floodsense.mapping import plot_raster_preview

class_raster = Path(paths['processed_rasters']) / 'susceptibility_class.tif'
if class_raster.exists():
    plot_raster_preview(class_raster, Path(paths['outputs_maps']) / 'susceptibility_notebook_preview.png', 'Flood Susceptibility Classes')
    print('Preview saved.')
else:
    print('Susceptibility class raster missing.')

## What to check before moving on

- Do High and Very High zones align with low-lying/river-proximate terrain?
- Are the classes too broad or too narrow?
- If results look unrealistic, review factor rasters and weights before validation.

Next notebook: `04_rainfall_scenario_analysis.ipynb`.